# 08 - Ingeniería de variables

Construye las variables derivadas que alimentan los modelos, a partir del dataset con valores ausentes tratados.

## Bloques de variables

1. **Objetivo a distintos horizontes**: valor del porcentaje de llenado a 7, 30 y 90 días.
2. **Retardos y medias móviles** de la variable objetivo y de las predictoras.
3. **Acumulados de precipitación** a distintas ventanas temporales.
4. **Evapotranspiración de referencia** por el método de Hargreaves-Samani.
5. **Variables de calendario**, con codificación cíclica de la estacionalidad.

## Criterio de construcción

Toda variable asociada a una fecha debe emplear exclusivamente información disponible en esa fecha o anterior. Las medias móviles se calculan con ventana cerrada por la derecha y los retardos se aplican sobre cada embalse de forma independiente, de modo que ninguna serie tome valores de otra.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

sys.path.append("..")

DIR_PROCESSED = Path("../data/processed")

HORIZONTES = [7, 30, 90]
LAGS = [1, 7, 30]
VENTANAS = [7, 30, 90]

dataset = (pd.read_parquet(DIR_PROCESSED / "dataset_imputado.parquet")
           .sort_values(["ID_SAIH", "fecha"])
           .reset_index(drop=True))

experimentales = set(pd.read_parquet(
    DIR_PROCESSED / "embalses_experimento_calidad.parquet")["ID_SAIH"])

print(f"Dataset: {len(dataset):,} filas | {dataset['ID_SAIH'].nunique()} embalses")
print(f"Columnas: {list(dataset.columns)}")

Dataset: 223,516 filas | 34 embalses
Columnas: ['fecha', 'ID_SAIH', 'volumen_hm3', 'pct_llenado', 'aportacion_m3s', 'salida_m3s', 'aemet_temp_media_c', 'aemet_temp_min_c', 'aemet_temp_max_c', 'aemet_precipitacion_mm', 'aemet_humedad_pct', 'cal_arriba_amonio_mgl', 'cal_arriba_conductividad_uscm', 'cal_arriba_oxigeno_mgl', 'cal_arriba_ph', 'cal_arriba_temp_agua_c', 'cal_arriba_turbidez_ntu', 'cal_abajo_amonio_mgl', 'cal_abajo_conductividad_uscm', 'cal_abajo_oxigeno_mgl', 'cal_abajo_ph', 'cal_abajo_temp_agua_c', 'cal_abajo_turbidez_ntu', 'Nombre_SAIH', 'Sistema', 'Capacidad_hm3']


## 1. Variable objetivo a distintos horizontes

Se construyen tres variables objetivo, correspondientes al porcentaje de llenado observado 7, 30 y 90 días después de la fecha de cada registro. El desplazamiento se aplica por embalse, de modo que ninguna serie tome valores de otra.

Las filas del final de cada serie quedan sin objetivo definido, al no existir observación futura disponible, y se descartan en la fase de modelado.

In [2]:
df = dataset.copy()

for h in HORIZONTES:
    df[f"objetivo_h{h}"] = df.groupby("ID_SAIH")["pct_llenado"].shift(-h)

print("Cobertura de las variables objetivo:")
for h in HORIZONTES:
    c = f"objetivo_h{h}"
    print(f"  {c:<16} {df[c].notna().mean()*100:5.1f}%  "
          f"({df[c].isna().sum():,} sin valor)")

Cobertura de las variables objetivo:
  objetivo_h7       97.1%  (6,551 sin valor)
  objetivo_h30      96.7%  (7,287 sin valor)
  objetivo_h90      95.9%  (9,207 sin valor)


### Nota sobre el uso de las variables objetivo

Las variables `objetivo_h7`, `objetivo_h30` y `objetivo_h90` responden a una formulación de regresión directa, en la que cada observación asocia las predictoras de una fecha con el valor observado h días después. Este planteamiento es el que emplean los modelos de aprendizaje automático (XGBoost y LSTM).

Los modelos SARIMAX, por su naturaleza autorregresiva, no emplean estas variables: se ajustan sobre la serie original del porcentaje de llenado y el horizonte se implementa en la fase de previsión. La diferencia de formulación entre familias de modelos no afecta al diseño experimental, ya que la comparación entre escenarios se realiza siempre dentro de un mismo modelo y horizonte.

## 2. Retardos y medias móviles

Se incorporan retardos de 1, 7 y 30 días y medias móviles de 7, 30 y 90 días sobre la variable objetivo y las principales predictoras hidrológicas y meteorológicas.

Las medias móviles se calculan con ventana cerrada por la derecha: el valor asignado a una fecha resume el intervalo que termina en ella, sin incorporar observaciones posteriores. Esta precaución es necesaria para que las variables derivadas no introduzcan información no disponible en el momento de la predicción.

In [3]:
BASE_DERIVADAS = ["pct_llenado", "aportacion_m3s", "salida_m3s",
                  "aemet_precipitacion_mm", "aemet_temp_media_c"]

g = df.groupby("ID_SAIH")

for var in BASE_DERIVADAS:
    for lag in LAGS:
        df[f"{var}_lag{lag}"] = g[var].shift(lag)
    for v in VENTANAS:
        # min_periods igual a la ventana: no se calcula la media hasta disponer
        # del intervalo completo, evitando valores basados en pocos días
        df[f"{var}_ma{v}"] = g[var].transform(
            lambda s, v=v: s.rolling(window=v, min_periods=v).mean())

nuevas = [c for c in df.columns if "_lag" in c or "_ma" in c]
print(f"Variables creadas: {len(nuevas)}")
print(f"Cobertura mínima: {df[nuevas].notna().mean().min()*100:.1f}%")

Variables creadas: 31
Cobertura mínima: 94.6%


## 3. Acumulados de precipitación

La precipitación acumulada en los días previos resulta más informativa que el registro diario aislado, al reflejar el aporte hídrico reciente sobre la cuenca vertiente. Se calculan acumulados a 3, 7, 30 y 90 días, así como el número de días sin precipitación en la ventana de 30 días, indicador de periodos secos prolongados.

In [4]:
VENTANAS_PREC = [3, 7, 30, 90]
UMBRAL_DIA_SECO = 0.1  # mm

for v in VENTANAS_PREC:
    df[f"prec_acum{v}"] = g["aemet_precipitacion_mm"].transform(
        lambda s, v=v: s.rolling(window=v, min_periods=v).sum())

df["dias_secos_30"] = g["aemet_precipitacion_mm"].transform(
    lambda s: (s < UMBRAL_DIA_SECO).rolling(window=30, min_periods=30).sum())

print(df[[f"prec_acum{v}" for v in VENTANAS_PREC] + ["dias_secos_30"]]
      .describe().round(1).to_string())

       prec_acum3  prec_acum7  prec_acum30  prec_acum90  dias_secos_30
count    223448.0    223312.0     222530.0     220490.0       222530.0
mean          8.3        19.3         82.6        246.0           16.6
std          14.7        27.3         75.7        165.0            6.8
min           0.0         0.0          0.0         11.4            0.0
25%           0.0         0.8         27.3        122.0           12.0
50%           1.4         8.7         60.3        202.9           17.0
75%          10.5        26.4        115.1        332.5           22.0
max         197.2       325.2        811.2       1255.7           30.0


## 4. Evapotranspiración de referencia (Hargreaves-Samani)

La evaporación es una componente relevante del balance hídrico de un embalse. Descartada la serie de evaporación del Anuario de Aforos por su escasa cobertura, se estima la evapotranspiración de referencia mediante el método de Hargreaves-Samani, que requiere únicamente temperaturas y la latitud del emplazamiento:

$$ET_0 = 0{,}0023 \cdot R_a \cdot (T_{med} + 17{,}8) \cdot (T_{max} - T_{min})^{0{,}5}$$

donde $R_a$ es la radiación extraterrestre, calculada en función de la latitud y del día del año según la formulación recogida en el manual FAO-56 (Allen et al., 1998).

El método se selecciona por su bajo requerimiento de datos: la alternativa de referencia, Penman-Monteith, exige radiación neta, velocidad del viento y humedad relativa, variables que no están disponibles con cobertura suficiente en el conjunto de embalses analizado.

In [5]:
from pyproj import Transformer

maestro = pd.read_parquet(DIR_PROCESSED / "maestro_embalses.parquet")

# Latitud de cada embalse, necesaria para la radiación extraterrestre
transformer = Transformer.from_crs("EPSG:25829", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(maestro["X"].values, maestro["Y"].values)
latitudes = dict(zip(maestro["ID_SAIH"], lat))

print(f"Latitudes: {min(lat):.2f} a {max(lat):.2f} grados")

Latitudes: 41.93 a 42.91 grados


In [6]:
def radiacion_extraterrestre(lat_grados, dia_juliano):
    """Radiación extraterrestre diaria (MJ m-2 d-1), según FAO-56 (Allen et al., 1998)."""
    lat = np.radians(lat_grados)
    # Distancia relativa inversa Tierra-Sol y declinación solar
    dr = 1 + 0.033 * np.cos(2 * np.pi * dia_juliano / 365.25)
    decl = 0.409 * np.sin(2 * np.pi * dia_juliano / 365.25 - 1.39)
    # Ángulo horario al ocaso
    arg = np.clip(-np.tan(lat) * np.tan(decl), -1, 1)
    ws = np.arccos(arg)
    return (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(lat) * np.sin(decl) + np.cos(lat) * np.cos(decl) * np.sin(ws))

df["lat"] = df["ID_SAIH"].map(latitudes)
dia_juliano = df["fecha"].dt.dayofyear

df["radiacion_ext"] = radiacion_extraterrestre(df["lat"].values, dia_juliano.values)

amplitud = (df["aemet_temp_max_c"] - df["aemet_temp_min_c"]).clip(lower=0)

# Ra se expresa en MJ m-2 d-1; la fórmula de Hargreaves-Samani requiere su
# equivalente en mm/día, dividiendo por el calor latente de vaporización (2,45 MJ/kg)
LATENTE = 2.45

df["et0_mm"] = (0.0023 * (df["radiacion_ext"] / LATENTE)
                * (df["aemet_temp_media_c"] + 17.8)
                * np.sqrt(amplitud))

print(f"ET0: cobertura {df['et0_mm'].notna().mean()*100:.1f}%")
print(df.groupby(df["fecha"].dt.month)["et0_mm"].mean().round(2).to_string())
print(f"\nET0 anual media: {df.groupby(df['fecha'].dt.year)['et0_mm'].mean().mean()*365.25:.0f} mm")

ET0: cobertura 100.0%
fecha
1     0.86
2     1.41
3     2.29
4     3.33
5     4.41
6     5.11
7     5.67
8     5.13
9     3.70
10    2.16
11    1.07
12    0.77

ET0 anual media: 1096 mm


In [7]:
anual_emb = (df.groupby(["ID_SAIH", df["fecha"].dt.year])["et0_mm"].sum()
             .groupby("ID_SAIH").mean().round(0))
print(anual_emb.sort_values().to_string())

ID_SAIH
E003     958.0
E16A     983.0
E022    1005.0
E021    1006.0
E017    1007.0
E05A    1031.0
E35A    1047.0
E023    1056.0
E013    1056.0
E026    1064.0
E025    1064.0
E18A    1067.0
E001    1067.0
E014    1069.0
E015    1069.0
E36A    1076.0
E570    1078.0
E024    1083.0
E028    1092.0
E020    1105.0
E19A    1105.0
E033    1116.0
E32A    1120.0
E07A    1148.0
E031    1148.0
E008    1148.0
E009    1152.0
E350    1161.0
E011    1161.0
E571    1174.0
E030    1177.0
E002    1223.0
E029    1223.0
E027    1224.0


### Limitación de la estimación

La evapotranspiración de referencia se calcula a partir de las temperaturas registradas en las estaciones de AEMET asignadas a cada embalse, cuya altitud no siempre se corresponde con la del emplazamiento. En los embalses de cabecera, situados por encima de los 1.000 m, las estaciones disponibles se ubican en cotas inferiores, lo que conduce a una sobreestimación de la temperatura y, por tanto, de la ET₀.

El efecto se aprecia al comparar los valores anuales por embalse: Cenza, a 1.341 m, y Frieira, a 66 m, arrojan estimaciones prácticamente idénticas pese a la diferencia de cota. La variable resulta por tanto fiable en su evolución temporal dentro de cada serie, pero no en la comparación de niveles entre embalses.

Para los modelos multi-embalse se incorpora una versión tipificada por embalse, que expresa la desviación respecto al comportamiento habitual de cada emplazamiento en lugar del valor absoluto. Los estadísticos de tipificación se calculan exclusivamente sobre el periodo de entrenamiento.

Cabe señalar además que el método de Hargreaves-Samani tiende a sobreestimar la evapotranspiración en climas húmedos, lo que sitúa las estimaciones obtenidas por encima de los valores de referencia habituales para el noroeste peninsular.

In [8]:
FIN_TRAIN = pd.Timestamp("2021-12-31")

# Estadísticos calculados solo sobre entrenamiento, para no incorporar
# información del periodo de test en la tipificación
train = df[df["fecha"] <= FIN_TRAIN]
stats_et0 = train.groupby("ID_SAIH")["et0_mm"].agg(["mean", "std"])

df["et0_tipificada"] = (
    (df["et0_mm"] - df["ID_SAIH"].map(stats_et0["mean"]))
    / df["ID_SAIH"].map(stats_et0["std"])
)

print(df.groupby("ID_SAIH")["et0_tipificada"].agg(["mean", "std"]).describe().round(3).to_string())

         mean     std
count  34.000  34.000
mean    0.007   1.003
std     0.003   0.003
min     0.002   0.999
25%     0.006   1.002
50%     0.006   1.003
75%     0.008   1.004
max     0.013   1.009


## 5. Variables de calendario

El llenado de un embalse presenta un marcado componente estacional, asociado tanto al régimen de precipitaciones como a la gestión de la explotación. Se incorporan el mes y el día del año, este último con codificación cíclica mediante funciones seno y coseno, de modo que el 31 de diciembre y el 1 de enero resulten próximos en el espacio de variables en lugar de situarse en extremos opuestos.

In [9]:
dia = df["fecha"].dt.dayofyear

df["mes"] = df["fecha"].dt.month
df["dia_anio_sin"] = np.sin(2 * np.pi * dia / 365.25)
df["dia_anio_cos"] = np.cos(2 * np.pi * dia / 365.25)

# Año hidrológico: comienza el 1 de octubre
df["anio_hidrologico"] = np.where(df["fecha"].dt.month >= 10,
                                  df["fecha"].dt.year + 1, df["fecha"].dt.year)

print(df[["mes", "dia_anio_sin", "dia_anio_cos", "anio_hidrologico"]].describe().round(2).to_string())

             mes  dia_anio_sin  dia_anio_cos  anio_hidrologico
count  223516.00     223516.00     223516.00         223516.00
mean        6.52          0.00         -0.00           2014.75
std         3.45          0.71          0.71              5.21
min         1.00         -1.00         -1.00           2006.00
25%         4.00         -0.70         -0.71           2010.00
50%         7.00         -0.00          0.00           2015.00
75%        10.00          0.71          0.70           2019.00
max        12.00          1.00          1.00           2024.00


## 6. Guardado

Se guarda el dataset con las variables derivadas, entrada de la fase de modelado.

In [10]:
df.to_parquet(DIR_PROCESSED / "dataset_modelado.parquet", index=False)

derivadas = [c for c in df.columns if c not in dataset.columns]
print(f"Guardado: {len(df):,} filas x {df.shape[1]} columnas")
print(f"Variables derivadas: {len(derivadas)}")
print(f"\nCobertura de las derivadas: mínima {df[derivadas].notna().mean().min()*100:.1f}%")

Guardado: 223,516 filas x 72 columnas
Variables derivadas: 46

Cobertura de las derivadas: mínima 94.6%
